In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_esquema = "movie_gold"
v_tabla = "result_group_movie_country"
v_partition = "file_date"

In [0]:
#Extracion de datos
##Tabla de carga completa
countries_df = spark.read.table("movie_silver.countries")

#Tablas con data particionada
movies_df = spark.read.table("movie_silver.movies")\
                      .filter(
                               (col("file_date") == f"{v_file_date}")
                             )
##                      
production_countries_df = spark.read.table("movie_silver.productions_countries")\
                                    .filter(
                                             (col("file_date") == f"{v_file_date}")
                                           )


movies_df.show(4)
production_countries_df.show(4)
countries_df.show(4)

In [0]:
#Tabla de movies filtrada con los campos y datos que necesitamos

movies_df = movies_df.select(movies_df.movie_id,
                            movies_df.year_release_date, 
                            movies_df.budget,
                            movies_df.revenue
                            )
movies_df.show(4)

In [0]:
#eneramos la tabla agregada con los campos solicitados
country_production_df = countries_df.join( production_countries_df,
                                           countries_df.country_id == production_countries_df.country_id,
                                          "inner"
                               )\
                          .select(production_countries_df.movie_id, countries_df.country_name)


movies_country_df = movies_df.join(country_production_df,
                                    movies_df.movie_id == country_production_df.movie_id,
                                    "inner"
                                       )\
                              .select(movies_df["*"], country_production_df.country_name)


In [0]:
movies_country_df.show(4)

In [0]:

movies_country_agg_df = movies_country_df.groupBy("year_release_date", "country_name")\
                                         .agg(
                                              sum("budget").alias("sum_budget"),
                                              sum("revenue").alias("sum_revenue")   
                                            )

movies_country_agg_df.show(3)                                                    

In [0]:
#"movies_genre_agg_df.select("year_release_date")
result_group_movie_country_df = movies_country_agg_df.select( "year_release_date","country_name", "sum_budget", "sum_revenue")\
                                                     .withColumn("dense_rank", dense_rank().over(Window.partitionBy("year_release_date")
                                                                                                       .orderBy(desc("sum_budget"))
                                                                                                       .orderBy(desc("sum_revenue"))
                                                                                                 )
                                                                )
                                        
result_group_movie_country_df.display()

In [0]:

result_group_movie_country_df = add_ingestion_date(result_group_movie_country_df)
result_group_movie_country_df = add_env(result_group_movie_country_df)
result_group_movie_country_df = add_file_date (result_group_movie_country_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
overwrite_partition (v_esquema, v_tabla, v_partition, v_file_date)

In [0]:
#Guardamos en la capa gold 

result_group_movie_country_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {result_group_movie_country_df.count()} registros en la tabla {v_esquema}.{v_tabla}")

